In [ ]:
'''
Santa 2026 | Verified Frontier 128,164

Replay the public portfolio and pinned local improvements and export the verified submission.
'''

In [ ]:
'''
Credits

The pinned 128,169 public portfolio combines programs from goodpjw2008 and Ertuğrul Özer.
https://www.kaggle.com/code/goodpjw2008/santa-2026-fresh-program-re-search-lb-128-170
https://www.kaggle.com/code/ozertuu/santa-2026-multi-engine-frontier-lb-128-169

The shorter scored_0164 program comes from Ertuğrul Özer.
https://www.kaggle.com/code/ozertuu/santa-2026-multi-engine-frontier-lb-128-168
The shorter scored_0219 program comes from Daniel Korbelainen.
https://www.kaggle.com/code/pand999/santa-2026-verified-frontier-128-168

Local searches with goodpjw2008's triwalk engine shortened scored_0872 and scored_0881.
The exact calculator replay builds on Ertuğrul Özer's Starter Notebook.
https://www.kaggle.com/code/ozertuu/santa-2026-starter-notebook
'''

# Santa 2026 · Verified Frontier

**128,164 arithmetic instructions across 1,000 puzzles.** Four replacements save five instructions against the pinned public portfolio. Every program is replayed on the 48-register calculator before the notebook writes `submission.csv`.

| Puzzle | Public baseline | Verified length | Source |
|:---|---:|---:|:---|
| `scored_0164` | 111 | 110 | Ertuğrul Özer |
| `scored_0219` | 107 | 106 | Daniel Korbelainen |
| `scored_0872` | 102 | 100 | Local search from the current best program |
| `scored_0881` | 81 | 80 | Local search from an archived program |
| **Portfolio** | **128,169** | **128,164** | |

The public portfolio is embedded as a checksum-verified snapshot. On Kaggle, attach the Santa 2026 competition data and run all cells.

## 1 · Inputs and configuration

In [ ]:
import csv
import hashlib
import os
import json
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from statistics import median
from tempfile import TemporaryDirectory
from time import perf_counter


COMPETITION = 'santa-2026'
PORTFOLIO_SHA256 = '8ceebbf3c748a6be4d47d51120716ef5f16ed929c0584cbc9ed5cce5b808ccba'
EXPECTED_SCORE = 128_164
SUBMISSION_COLUMNS = ['id', 'program']
MAX_INSTRUCTIONS = 4_096
MAX_PROGRAM_BYTES = 131_072

Instruction = tuple[str, int, int, int]
Program = tuple[Instruction, ...]


@dataclass(frozen=True)
class Config:
    puzzles_csv: Path
    output_csv: Path

    @classmethod
    def discover(cls) -> 'Config':
        kaggle = Path('/kaggle/input')
        if kaggle.is_dir():
            puzzles = cls.locate(kaggle, COMPETITION, 'puzzles.csv')
            return cls(puzzles, Path('/kaggle/working/submission.csv'))

        data = Path(os.environ.get('SANTA_DATA_DIR', 'data'))
        return cls(data / 'puzzles.csv',
                   Path(os.environ.get('SANTA_OUTPUT_CSV', 'submission.csv')))

    @staticmethod
    def locate(root: Path, slug: str, filename: str) -> Path:
        matches = [path for path in root.rglob(filename) if slug in path.parent.parts]
        if len(matches) != 1:
            raise FileNotFoundError(
                f'Expected one {slug}/{filename} input, found {len(matches)}'
            )
        return matches[0]

    def check_inputs(self) -> None:
        if not self.puzzles_csv.is_file():
            raise FileNotFoundError(
                f'Attach the {COMPETITION} competition data or copy puzzles.csv '
                f'to {self.puzzles_csv}'
            )

In [ ]:
'''
Embed the public portfolio so each run uses the same programs.
'''
import base64
import lzma


PUBLIC_PORTFOLIO_XZ = (
    '...'
    '334F4SpSscRn+wIAAAAABFla'
)

def pinned_portfolio() -> bytes:
    raw = lzma.decompress(base64.b64decode(PUBLIC_PORTFOLIO_XZ, validate=True))
    if hashlib.sha256(raw).hexdigest() != PORTFOLIO_SHA256:
        raise ValueError('The embedded public portfolio failed its checksum')
    return raw


## 2 · Exact calculator and program profile

In [ ]:
@dataclass(frozen=True)
class Puzzle:
    starts: tuple[int, ...]
    targets: frozenset[int]
    value_limit: int
    register_count: int

    @classmethod
    def from_row(cls, row: dict[str, str]) -> 'Puzzle':
        return cls(
            tuple(json.loads(row['starting_values'])),
            frozenset(json.loads(row['targets'])),
            int(row['value_limit']),
            int(row['registers']),
        )


@dataclass(frozen=True)
class ProgramProfile:
    length: int
    operations: Counter[str]
    peak_value: int
    populated_registers: int


class Calculator:
    '''Read both sources before writing and remember targets across overwrites.'''

    def replay(self, puzzle: Puzzle, program: Program) -> ProgramProfile:
        if len(program) > MAX_INSTRUCTIONS:
            raise ValueError('Program exceeds the instruction limit')

        registers = list(puzzle.starts) + [None] * (
            puzzle.register_count - len(puzzle.starts)
        )
        missing = set(puzzle.targets).difference(puzzle.starts)
        operations = Counter()
        peak_value = max(puzzle.starts, default=0)

        for number, instruction in enumerate(program, start=1):
            if len(instruction) != 4:
                raise ValueError(f'Step {number}: expected four instruction fields')
            op, left, right, dest = instruction
            if any(type(index) is not int or not 0 <= index < puzzle.register_count
                   for index in (left, right, dest)):
                raise ValueError(f'Step {number}: register index out of range')

            a, b = registers[left], registers[right]
            if a is None or b is None:
                raise ValueError(f'Step {number}: undefined source register')
            if op == '+':
                value = a + b
            elif op == '-':
                value = a - b
            elif op == '*':
                value = a * b
            elif op == '/' and a % b == 0:
                value = a // b
            else:
                raise ValueError(f'Step {number}: unsupported operation or inexact division')

            if not 1 <= value <= puzzle.value_limit:
                raise ValueError(f'Step {number}: result outside the value limit')
            registers[dest] = value
            missing.discard(value)
            operations[op] += 1
            peak_value = max(peak_value, value)

        if missing:
            raise ValueError(f'{len(missing)} missing targets')

        '''
        Defined registers stay populated after a write, so the final count
        is also the peak populated count for the program.
        '''
        return ProgramProfile(
            len(program), operations, peak_value,
            sum(value is not None for value in registers),
        )

## 3 · Validate the portfolio and export

In [ ]:
PATCHES = {
    'scored_0164': (111, '[["-",1,0,3],["+",0,2,4],["+",2,4,5],["*",3,5,6],["+",3,1,7],["+",7,7,8],["*",8,7,9],["+",1,8,10],["-",9,7,11],["-",11,8,12],["*",1,7,13],["*",3,13,14],["*",8,14,15],["+",14,11,11],["*",3,11,16],["*",3,16,17],["+",3,17,18],["+",17,16,19],["+",13,16,16],["-",5,1,20],["*",0,20,21],["+",0,21,22],["+",22,15,23],["*",18,23,24],["*",21,9,24],["-",24,9,25],["-",25,20,25],["*",5,20,26],["+",3,26,27],["*",7,26,28],["*",0,28,29],["/",29,6,30],["*",0,30,31],["*",31,24,32],["+",28,32,32],["-",32,19,32],["+",31,23,32],["-",32,26,32],["*",3,32,33],["*",33,16,16],["*",33,24,24],["+",28,24,24],["+",23,32,23],["-",16,23,16],["*",19,32,23],["*",11,23,24],["+",25,29,24],["-",24,9,24],["*",27,24,32],["/",32,1,34],["-",32,27,27],["-",16,30,16],["*",0,29,27],["*",27,14,14],["-",14,31,14],["-",27,3,3],["-",3,25,14],["+",33,3,3],["*",3,4,4],["-",4,9,4],["+",26,6,4],["*",4,15,6],["-",6,3,6],["*",11,6,6],["+",1,4,6],["-",29,4,4],["*",18,4,14],["-",20,2,2],["+",9,2,9],["+",9,4,4],["-",4,28,4],["*",11,4,15],["*",9,14,14],["+",9,30,18],["+",18,18,20],["*",20,4,4],["-",15,20,15],["-",31,18,15],["+",15,3,3],["*",15,14,3],["+",15,7,3],["*",3,6,6],["-",6,3,7],["-",6,17,14],["+",6,14,6],["-",24,18,6],["*",28,6,6],["*",1,6,1],["-",1,31,1],["+",18,19,1],["*",1,13,6],["+",0,6,0],["*",17,6,0],["+",0,10,0],["*",5,1,0],["*",15,0,1],["+",11,1,1],["*",0,2,0],["-",0,8,1],["+",0,10,0],["-",0,7,0],["+",1,12,0],["-",23,0,2],["+",4,2,4],["*",21,0,0],["-",1,9,0],["*",0,22,0],["-",0,3,0],["-",2,20,0],["+",0,16,0]]'),
    'scored_0219': (107, '[["-",2,1,4],["+",4,0,5],["+",5,3,6],["-",5,1,7],["/",3,7,8],["+",3,8,9],["*",3,9,10],["+",0,10,11],["-",9,1,12],["*",1,12,13],["+",9,13,14],["-",13,6,15],["*",6,15,15],["*",6,15,16],["*",0,16,17],["*",14,16,18],["-",18,11,19],["*",9,18,18],["*",0,1,19],["-",19,8,20],["*",20,18,21],["*",20,14,22],["*",10,22,23],["+",20,13,24],["*",6,20,25],["-",25,24,26],["*",26,26,27],["*",27,14,28],["-",27,4,28],["-",28,1,29],["+",29,12,30],["*",5,29,29],["*",0,28,30],["-",30,5,30],["*",26,15,15],["+",15,6,30],["/",25,1,31],["+",13,31,32],["+",32,32,33],["+",33,27,27],["*",1,27,27],["*",33,28,27],["*",2,31,27],["-",21,27,21],["-",33,6,21],["*",1,20,27],["*",5,27,28],["+",18,28,18],["*",33,26,18],["*",18,23,23],["+",4,23,23],["-",18,25,18],["-",33,24,23],["+",23,29,26],["*",0,24,24],["*",33,17,26],["+",24,26,26],["-",24,32,26],["*",26,13,13],["-",13,21,29],["+",3,29,3],["*",9,3,3],["+",2,29,3],["-",3,22,3],["*",24,13,13],["+",13,23,13],["*",1,29,1],["+",5,1,1],["+",11,26,1],["*",10,1,10],["+",10,7,10],["*",2,10,2],["*",1,16,1],["/",1,5,5],["-",1,0,1],["*",20,1,13],["-",1,0,1],["*",4,1,1],["-",1,8,1],["+",5,21,1],["-",18,10,1],["-",2,22,1],["+",28,1,1],["*",7,10,1],["-",30,12,2],["-",2,31,2],["+",15,2,4],["-",2,25,2],["-",2,3,2],["*",2,8,3],["-",3,23,3],["-",2,20,2],["*",9,2,3],["-",3,14,3],["-",3,21,3],["*",2,27,2],["+",11,19,2],["-",1,2,1],["+",6,2,1],["-",1,12,2],["*",2,5,2],["+",0,2,2],["*",32,2,2],["-",17,1,1],["*",0,17,0],["-",0,32,0]]'),
    'scored_0872': (102, '[["*",1,3,3],["+",3,3,5],["-",2,0,6],["-",1,0,7],["*",7,4,8],["*",7,7,9],["+",1,3,10],["+",2,4,11],["+",2,11,12],["*",12,9,13],["+",3,13,13],["*",0,11,14],["+",2,14,15],["+",15,5,16],["-",15,4,17],["*",17,9,18],["+",16,9,9],["+",9,3,19],["+",19,8,20],["+",9,19,21],["*",16,21,22],["+",7,22,22],["+",19,21,22],["*",2,9,2],["-",2,13,23],["*",0,23,24],["+",16,24,25],["-",24,10,26],["*",17,24,17],["+",19,17,17],["-",22,1,17],["+",20,17,26],["*",6,22,27],["-",27,2,28],["*",7,28,29],["+",11,27,27],["*",6,29,30],["+",8,27,31],["*",19,31,19],["-",20,4,19],["+",29,19,32],["+",32,18,18],["-",24,19,32],["+",17,32,32],["-",32,31,33],["*",26,33,26],["+",28,26,26],["*",18,13,13],["-",13,4,13],["-",13,29,13],["*",15,19,13],["-",13,5,5],["+",8,13,5],["-",5,3,3],["*",6,5,8],["-",8,17,8],["*",7,32,13],["*",1,13,15],["-",15,10,15],["*",1,15,1],["-",15,10,10],["+",13,10,10],["+",19,31,10],["+",21,5,5],["-",5,10,10],["*",4,5,4],["*",10,9,9],["-",9,22,9],["-",5,2,5],["*",25,5,9],["-",5,11,5],["-",3,19,3],["-",18,2,2],["-",2,14,5],["*",7,5,10],["*",16,10,11],["+",0,10,11],["+",11,33,13],["+",13,12,12],["-",30,12,12],["*",12,20,13],["+",13,27,14],["+",12,1,1],["*",7,10,1],["+",1,23,7],["+",1,15,1],["-",1,6,6],["+",4,6,4],["+",13,1,1],["+",28,11,1],["-",1,24,1],["+",19,2,1],["*",17,1,2],["-",2,10,2],["+",4,9,2],["+",1,2,1],["+",3,3,1],["-",1,0,0],["-",8,5,0],["+",1,0,0]]'),
    'scored_0881': (81, '[["*",0,1,4],["-",2,0,2],["-",3,2,5],["*",2,4,6],["+",2,6,7],["*",4,7,8],["/",8,2,9],["+",5,8,10],["+",10,6,11],["+",10,11,12],["-",9,6,13],["-",13,1,1],["+",1,12,14],["-",7,1,14],["*",14,10,15],["*",5,1,15],["/",15,0,16],["-",16,7,16],["*",13,16,17],["*",5,16,18],["+",13,18,18],["/",16,0,16],["-",18,2,19],["+",19,10,20],["*",3,19,20],["*",16,20,20],["*",14,19,21],["+",19,19,22],["+",15,22,23],["+",13,19,24],["-",16,5,16],["-",22,16,22],["+",10,22,10],["-",19,7,7],["+",4,16,10],["-",22,10,22],["*",4,10,22],["-",22,7,7],["-",7,10,25],["+",25,15,15],["*",4,15,4],["+",13,7,4],["*",4,9,4],["-",15,3,3],["-",3,14,9],["*",5,7,13],["-",13,11,11],["-",11,24,13],["+",0,11,0],["+",11,21,0],["-",4,17,0],["*",0,18,0],["+",16,1,0],["+",8,0,1],["+",1,2,1],["*",1,22,2],["+",0,9,2],["*",5,2,2],["*",6,0,0],["*",5,13,2],["+",23,2,4],["+",16,2,2],["*",10,2,4],["-",2,3,2],["-",2,7,3],["-",3,22,3],["+",1,2,1],["-",1,18,1],["+",1,12,2],["*",7,1,2],["-",2,17,2],["+",24,2,2],["+",22,0,0],["-",0,11,2],["-",2,19,3],["+",0,2,2],["-",0,17,0],["+",8,0,0],["+",0,1,0],["+",20,0,0]]'),
}


class PortfolioPipeline:
    def __init__(self, config: Config):
        self.config = config
        self.calculator = Calculator()

    def load_puzzles(self) -> dict[str, Puzzle]:
        with self.config.puzzles_csv.open(encoding='utf-8-sig', newline='') as file:
            rows = list(csv.DictReader(file))
        puzzles = {row['id']: Puzzle.from_row(row) for row in rows}
        expected = {f'scored_{index:04d}' for index in range(1, 1_001)}
        if len(rows) != len(expected) or puzzles.keys() != expected:
            raise ValueError('Expected the 1,000 scored puzzle IDs exactly once')
        return puzzles

    def verify(
        self, path: Path, puzzles: dict[str, Puzzle],
    ) -> tuple[dict[str, Program], dict[str, ProgramProfile]]:
        programs, profiles = {}, {}
        with path.open(encoding='utf-8', newline='') as file:
            reader = csv.reader(file)
            if next(reader, None) != SUBMISSION_COLUMNS:
                raise ValueError(f'{path}: expected the id,program header')
            for row in reader:
                if len(row) != 2:
                    raise ValueError(f'{path}: expected two fields at line {reader.line_num}')
                puzzle_id, cell = row
                if puzzle_id not in puzzles or puzzle_id in programs:
                    raise ValueError(f'{path}: unexpected or duplicate ID {puzzle_id}')
                if len(cell.encode('utf-8')) > MAX_PROGRAM_BYTES:
                    raise ValueError(f'{puzzle_id}: program cell exceeds the byte limit')
                try:
                    payload = json.loads(cell)
                    if type(payload) is not list or any(type(step) is not list for step in payload):
                        raise ValueError('Expected a JSON array of instruction arrays')
                    program = tuple(tuple(step) for step in payload)
                    profiles[puzzle_id] = self.calculator.replay(puzzles[puzzle_id], program)
                except ValueError as error:
                    raise ValueError(f'{puzzle_id}: {error}') from error
                programs[puzzle_id] = program
        if programs.keys() != puzzles.keys():
            raise ValueError(f'{path}: portfolio is missing scored puzzle IDs')
        return programs, profiles

    def export(self, programs: dict[str, Program], puzzles: dict[str, Puzzle]) -> None:
        output = self.config.output_csv
        output.parent.mkdir(parents=True, exist_ok=True)
        with TemporaryDirectory(dir=output.parent) as directory:
            temporary = Path(directory) / output.name
            with temporary.open('w', encoding='utf-8', newline='') as file:
                writer = csv.writer(file)
                writer.writerow(SUBMISSION_COLUMNS)
                for puzzle_id in sorted(programs):
                    cell = json.dumps(programs[puzzle_id], separators=(',', ':'))
                    writer.writerow((puzzle_id, cell))
            written, _ = self.verify(temporary, puzzles)
            if written != programs:
                raise RuntimeError('The exported programs differ from the verified input')
            temporary.replace(output)

    def run(self) -> dict[str, ProgramProfile]:
        started = perf_counter()
        self.config.check_inputs()
        puzzles = self.load_puzzles()
        with TemporaryDirectory() as directory:
            source = Path(directory) / 'our_programs.csv'
            source.write_bytes(pinned_portfolio())
            programs, profiles = self.verify(source, puzzles)
        for pid, (original_length, encoded) in PATCHES.items():
            candidate = tuple(map(tuple, json.loads(encoded)))
            if len(programs[pid]) != original_length or len(candidate) >= original_length:
                raise ValueError(f'{pid}: the pinned patch does not shorten its reference')
            profiles[pid] = self.calculator.replay(puzzles[pid], candidate)
            programs[pid] = candidate
            print(f'{pid}: {original_length} → {len(candidate)}')
        lengths = [profile.length for profile in profiles.values()]
        score = sum(lengths)
        if score != EXPECTED_SCORE:
            raise ValueError(f'Unexpected portfolio score: {score:,}')
        self.export(programs, puzzles)

        operations = Counter()
        for profile in profiles.values():
            operations.update(profile.operations)
        print('Public baseline: 128,169 instructions')
        print(f'Verified puzzles: {len(programs):,} | Local score: {score:,}')
        print(f'Program lengths: min {min(lengths)} | median {median(lengths):g} | max {max(lengths)}')
        print('Operation counts:', ', '.join(f'{op} {operations[op]:,}' for op in '+-*/'))
        print(f'CSV: {self.config.output_csv} | {perf_counter() - started:.2f}s')
        return profiles

## 4 · Self-check and run

In [ ]:
calculator = Calculator()
puzzle = Puzzle((3,), frozenset({1, 2, 5, 8}), 9, 48)
example = (('/', 0, 0, 1), ('-', 0, 1, 1), ('+', 0, 1, 2), ('+', 0, 2, 3))
profile = calculator.replay(puzzle, example)
assert profile == ProgramProfile(4, Counter({'+': 2, '-': 1, '/': 1}), 8, 4)

try:
    calculator.replay(puzzle, (('/', 0, 0, 1), ('/', 1, 0, 2)))
except ValueError as error:
    assert 'inexact division' in str(error)
else:
    raise AssertionError('Inexact division passed replay')

profiles = PortfolioPipeline(Config.discover()).run()

## 5 · Where the instructions go

The left panel groups the verified instruction cost by number of targets. The right panel shows counts of executed operations. Operation frequency describes the portfolio and is not a measure of how much an operation could save.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter


class PortfolioFigures:
    @staticmethod
    def draw(puzzles, profiles):
        orange, blue = '#d9480f', '#1c7ed6'
        with plt.rc_context({
            'figure.facecolor': 'white', 'axes.spines.top': False,
            'axes.spines.right': False, 'axes.edgecolor': '#dee2e6',
            'axes.titlelocation': 'left', 'axes.titleweight': 'bold',
            'text.color': '#212529', 'grid.color': '#e9ecef',
        }):
            fig, (cost, operations) = plt.subplots(1, 2, figsize=(10.5, 3.7), layout='constrained')
            bands = [(1, 9), (10, 19), (25, 29), (30, 32)]
            totals = [
                sum(profile.length for pid, profile in profiles.items()
                    if low <= len(puzzles[pid].targets) <= high)
                for low, high in bands
            ]
            assert sum(totals) == sum(profile.length for profile in profiles.values())
            counts = Counter()
            for profile in profiles.values():
                counts.update(profile.operations)

            def bars(ax, labels, values, colors, title):
                rectangles = ax.barh(labels, values, color=colors, height=.6)
                ax.bar_label(rectangles, labels=[f'{value:,}' for value in values], padding=5)
                ax.set_xlim(0, max(values)*1.24)
                ax.xaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
                ax.set_title(title, pad=12)
                ax.set_xlabel('Executed instructions')
                ax.grid(axis='x')
                ax.set_axisbelow(True)

            bars(cost, [f'{a}–{b} targets' for a, b in bands], totals,
                 [blue, blue, orange, orange], 'Cost by target count')
            bars(operations, ['Addition', 'Subtraction', 'Multiplication', 'Exact division'],
                 [counts[op] for op in '+-*/'], [blue, blue, blue, orange], 'Operation mix')
            plt.show()


PortfolioFigures.draw(PortfolioPipeline(Config.discover()).load_puzzles(), profiles)

## 6 · Paired search

The pilot ran both starting programs on each of 24 puzzles, using the same seed within a pair and 90 CPU seconds per run. The current best program yielded a two-instruction improvement on `scored_0872`. The archived program for `scored_0881` started at 82 instructions and reached 80, one below the current best of 81. The notebook pins the two verified results.